In [2]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, DatasetDict
import torch
import re 
import pandas as pd

# Load pre-trained tokenizer and model
model_name = "neuralmind/bert-base-portuguese-cased"  # BERTimbau Base
tokenizer = BertTokenizer.from_pretrained(model_name)

# Define labels (11 categories from TuPy-E)
hate_labels = ['ageism', 'aporophobia', 'body_shame', 'capacitism', 'lgbtphobia',
               'political', 'racism', 'religious_intolerance', 'misogyny', 'xenophobia', 'other']


2025-05-02 18:39:18.967602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746211159.205652      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746211159.278513      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

In [3]:
from datasets import load_dataset

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    enc["labels"] = [float(example[label]) for label in hate_labels]
    return enc

tokenized_ds = ds.map(preprocess)


README.md:   0%|          | 0.00/8.28k [00:00<?, ?B/s]

multilabel_train.csv:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

multilabel_test.csv:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/34934 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8734 [00:00<?, ? examples/s]

Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

In [4]:
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=len(hate_labels), problem_type="multi_label_classification")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',  # Directory to save the model checkpoints
    eval_strategy="epoch",  # Evaluate after each epoch
    
    logging_strategy="no",  # Disables logging
    save_strategy="epoch",  # Save model checkpoint after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    num_train_epochs=20,  # Number of epochs to train
    per_device_train_batch_size=16,  # Batch size per device for training
    per_device_eval_batch_size=16,  # Batch size per device for evaluation
    learning_rate=2e-5,  # Learning rate for optimization
    weight_decay=0.01,  # Weight decay (for regularization)
    logging_dir='./logs',  # Directory to save logs
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model='eval_subset_accuracy',  # Metric to track the best model
    report_to=[],  # Don't report metrics to any logging service
)


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [5]:
from torch.utils.data import default_collate
import torch

class MultilabelDataCollator:
    def __call__(self, features):
        # Convert everything to tensors
        batch = {
            key: torch.tensor([f[key] for f in features])
            for key in features[0]
        }
        return batch

collator = MultilabelDataCollator()

from sklearn.metrics import f1_score, accuracy_score, classification_report, precision_score, recall_score

def compute_metrics(pred):
    logits, labels = pred.predictions, pred.label_ids
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    pred_labels = (probs >= 0.5).astype(int)

    # Subset accuracy (exact match across all labels)
    subset_acc = accuracy_score(labels, pred_labels)

    # Micro F1 (better for imbalanced multilabel data)
    micro_f1 = f1_score(labels, pred_labels, average='micro', zero_division=0)
    micro_precision = precision_score(labels, pred_labels, average='micro', zero_division=0)
    micro_recall = recall_score(labels, pred_labels, average='micro', zero_division=0)

    return {
        'subset_accuracy': subset_acc,
        'micro_f1': micro_f1,
        'micro_precision': micro_precision,
        'micro_recall': micro_recall
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=MultilabelDataCollator(),
    compute_metrics=compute_metrics  # ← This is the key addition
)

/tmp/ipykernel_31/1073832255.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [6]:
trainer.train()

Epoch,Training Loss,Validation Loss,Subset Accuracy,Micro F1,Micro Precision,Micro Recall
1,No log,0.050734,0.838562,0.447501,0.591837,0.359763
2,No log,0.045493,0.852645,0.478462,0.703665,0.362460
3,No log,0.047501,0.847607,0.560645,0.601732,0.524811
4,No log,0.054176,0.844401,0.528597,0.598093,0.473571
5,No log,0.062565,0.850927,0.521545,0.638780,0.440669
6,No log,0.070791,0.842111,0.526844,0.581380,0.481661
7,No log,0.075306,0.843600,0.546233,0.580000,0.516181
8,No log,0.079997,0.843943,0.541060,0.587975,0.501079
9,No log,0.083651,0.852416,0.535150,0.628364,0.466019
10,No log,0.086016,0.849324,0.546543,0.613988,0.492449


TrainOutput(global_step=43680, training_loss=0.011728837289216318, metrics={'train_runtime': 9892.1266, 'train_samples_per_second': 70.63, 'train_steps_per_second': 4.416, 'total_flos': 4.596132175309824e+16, 'train_loss': 0.011728837289216318, 'epoch': 20.0})

In [8]:
from sklearn.metrics import classification_report
import numpy as np
import torch

# Predict on test set
predictions = trainer.predict(tokenized_ds["test"])
logits = predictions.predictions
probs = torch.sigmoid(torch.tensor(logits)).numpy()
pred_labels = (probs >= 0.5).astype(int)

# Add non_hate column: 1 if no hate labels predicted
non_hate_preds = (pred_labels.sum(axis=1) == 0).astype(int)
non_hate_true = (predictions.label_ids.sum(axis=1) == 0).astype(int)

# Append non_hate as the 12th label
pred_labels = np.concatenate([pred_labels, non_hate_preds[:, None]], axis=1)
true_labels = np.concatenate([predictions.label_ids, non_hate_true[:, None]], axis=1)

# Updated label names
full_labels = hate_labels + ["non_hate"]

# Classification report
print(classification_report(true_labels, pred_labels, target_names=full_labels, zero_division=0))


                       precision    recall  f1-score   support

               ageism       0.20      0.08      0.12        12
          aporophobia       0.50      0.21      0.30        14
           body_shame       0.80      0.57      0.67        63
           capacitism       0.11      0.09      0.10        11
           lgbtphobia       0.78      0.76      0.77       149
            political       0.62      0.53      0.57       230
               racism       0.58      0.38      0.46        56
religious_intolerance       0.44      0.24      0.31        17
             misogyny       0.71      0.59      0.65       335
           xenophobia       0.58      0.43      0.50        88
                other       0.61      0.35      0.44       879
             non_hate       0.90      0.95      0.93      7188

            micro avg       0.86      0.85      0.86      9042
            macro avg       0.57      0.43      0.48      9042
         weighted avg       0.85      0.85      0.84 

In [9]:
trainer.save_model("./my-bertimbau-hate-base-model-cat")
tokenizer.save_pretrained("./my-bertimbau-hate-base-model-cat")


('./my-bertimbau-hate-base-model-cat/tokenizer_config.json',
 './my-bertimbau-hate-base-model-cat/special_tokens_map.json',
 './my-bertimbau-hate-base-model-cat/vocab.txt',
 './my-bertimbau-hate-base-model-cat/added_tokens.json')

In [10]:
import shutil

# Zip the saved model and tokenizer directory
shutil.make_archive('/kaggle/working/my-bertimbau-hate-base-model-cat', 'zip', './my-bertimbau-hate-base-model-cat')



'/kaggle/working/my-bertimbau-hate-base-model-cat.zip'